# Workbook Upload Demo

This notebook mirrors `scripts/upload_results_example_data.py`.

It parses the example workbook, builds validated analyses and results, and optionally uploads the rows that are compatible with the current dev backend.

In [112]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [113]:
import math

# import os
import pathlib
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase.locations.io import LocationsAPI
from tqdm.notebook import tqdm as tqdm_notebook

from owi.metadatabase.results import (
    AnalysisDefinition,
    LifetimeDesignFrequencies,
    ResultsAPI,
    ResultsService,
    )
from owi.metadatabase.results.models import ResultSeries
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer
from owi.metadatabase.results.services import ApiResultsRepository

WORKSPACE_ROOT = Path.cwd().resolve().parent
DEFAULT_WORKBOOK = WORKSPACE_ROOT / "scripts" / "data" / "results-example-data.xlsx"
print(f"Using workbook at {pathlib.Path(DEFAULT_WORKBOOK).resolve()}")
DEFAULT_BASE_URL = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
POST_DATA = False


Using workbook at /home/pietro.dantuono@24SEA.local/Projects/SMARTLIFE/OWI-metadatabase-SDK/owi-metadatabase-results-sdk/scripts/data/results-example-data.xlsx


In [ ]:
BASE_URL = DEFAULT_BASE_URL
WORKBOOK = DEFAULT_WORKBOOK
PROJECTSITE = "Belwind"
TOKEN = ""  # os.getenv(TOKEN_ENV_VAR)
LIVE_UPLOAD = TOKEN is not None

print({"workbook": str(WORKBOOK), "projectsite": PROJECTSITE, "live_upload": LIVE_UPLOAD})


{'workbook': '/home/pietro.dantuono@24SEA.local/Projects/SMARTLIFE/OWI-metadatabase-SDK/owi-metadatabase-results-sdk/scripts/data/results-example-data.xlsx', 'projectsite': 'Belwind', 'live_upload': True}


In [84]:
def resolve_site_and_locations(
    api_root: str, token: str | None, projectsite: str, turbines: set[str], live: bool
) -> tuple[int | None, dict[str, int | None], pd.DataFrame]:
    if not live or token is None:
        return (
            1,
            dict.fromkeys(turbines),
            pd.DataFrame(columns=["id", "title", "northing", "easting"]),
        )
    locations_api = LocationsAPI(api_root=api_root, token=token)
    site_id = int(locations_api.get_projectsite_detail(projectsite=projectsite)["id"])

    assetlocations = locations_api.get_assetlocations(projectsite=projectsite)["data"]
    location_frame = assetlocations.loc[:, [
        column
        for column in ["id", "title", "northing", "easting"]
        if column in assetlocations.columns
    ]].copy()
    resolved_locations = {
        str(row["title"]): int(row["id"])

        for row in location_frame.to_dict(orient="records")
        if row.get("title") is not None and row.get("id") is not None
    }
    mapping = {turbine: resolved_locations.get(turbine) for turbine in sorted(turbines)}
    return site_id, mapping, location_frame

lifetime_design_frequencies_analysis = LifetimeDesignFrequencies()

def parse_frequencies_sheet(
    path: Path, site_id: int, location_map: dict[str, int | None]
) -> tuple[AnalysisDefinition, list[ResultSeries]]:
    frame = pd.read_excel(path, sheet_name="Lifetime -  Design frequencies")
    rows = []
    for row in frame.to_dict(orient="records"):
        rows.append(
            {
                "turbine": str(row["Turbine"]),
                "reference": row["Reference"],
                "FA1": row.get("FA1 [Hz]"),
                "SS1": row.get("SS1 [Hz]"),
                "FA2": row.get("FA2 [Hz]"),
                "SS2": row.get("SS2 [Hz]"),
                "site_id": site_id,
                "location_id": location_map.get(str(row["Turbine"])),
            }
        )

    analysis = AnalysisDefinition(
        name=lifetime_design_frequencies_analysis.analysis_name,
        source_type="notebook",
        description="Workbook upload for the lifetime design frequencies sheet.",
        additional_data={"sheet_name": "Lifetime -  Design frequencies"},
    )
    return analysis, LifetimeDesignFrequencies().to_results({"rows": rows})


In [85]:
verification_frame = pd.read_excel(WORKBOOK, sheet_name="Lifetime -  Design verification")
frequencies_frame = pd.read_excel(WORKBOOK, sheet_name="Lifetime -  Design frequencies")
turbines = set(verification_frame["Turbine"].astype(str)) | set(frequencies_frame["Turbine"].astype(str))
site_id, location_map, location_frame = resolve_site_and_locations(
    BASE_URL, TOKEN, PROJECTSITE, turbines, LIVE_UPLOAD
)  # pyright: ignore
if not site_id:
    print("No site ID could be resolved, cannot proceed with upload.")
    site_id: int = -1
display(pd.DataFrame([{"turbine": turbine, "location_id": location_map[turbine]}
                      for turbine in sorted(location_map)]).head())


,turbine,location_id
0,BBA01,435
1,BBA02,436
2,BBA03,437
3,BBA04,438
4,BBA05,439


In [86]:
payload_sets = [
    parse_frequencies_sheet(WORKBOOK, site_id, location_map),
]

summary_rows = [
    {
        "analysis": analysis_definition.name,
        "results": len(result_series),
        "sheet": analysis_definition.additional_data["sheet_name"],
    }
    for analysis_definition, result_series in payload_sets
]

display(pd.DataFrame(summary_rows))


,analysis,results,sheet
0,LifetimeDesignFrequencies,220,Lifetime - Design frequencies


In [87]:
analysis_serializer = DjangoAnalysisSerializer()
result_serializer = DjangoResultSerializer()

preview_rows = []
for analysis_definition, result_series in payload_sets:
    result_payloads = [result_serializer.to_payload(item, analysis_id=0) for item in result_series]
    invalid_payloads = [payload for payload in result_payloads if payload.get("location") is None]
    valid_payloads = [payload for payload in result_payloads if payload.get("location") is not None]
    first_valid_payload = valid_payloads[0] if valid_payloads else None
    preview_rows.append(
        {
            "analysis": analysis_definition.name,
            "uploadable_rows": len(valid_payloads),
            "skipped_rows": len(invalid_payloads),
            "first_result": result_series[0].short_description,
            "reference_storage": "additional_data.reference_labels",
            "stored_reference_labels": (
                first_valid_payload.get("additional_data", {}).get("reference_labels")
                if first_valid_payload is not None
                else []
            ),
        }
    )

display(pd.DataFrame(preview_rows))


,analysis,uploadable_rows,skipped_rows,first_result,reference_storage,stored_reference_labels
0,LifetimeDesignFrequencies,220,0,BBA01 - FA1,additional_data.reference_labels,"[INFL, ACTU, FLEX]"


## Data Upload

The next cell uploads only the rows that can be resolved to a backend `location` and have finite numeric values.

In [ ]:
api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
upload_log = []
for analysis_def, result_series in payload_sets:
    result_payloads = [result_serializer.to_payload(item, analysis_id=0)
                       for item in result_series]
    valid_payloads = [payload for payload in result_payloads
                      if payload.get("location") is not None]
    if not valid_payloads:
        upload_log.append({"analysis": analysis_def.name,
                           "uploaded_rows": 0, "status": "skipped"})
        continue

    created_analysis = api.create_analysis(
        analysis_serializer.to_payload(analysis_def)
    )
    analysis_id = int(created_analysis["id"])
    print(f"Created analysis with ID {analysis_id} for "
          f"{analysis_def.name}, uploading {len(valid_payloads)} results...")
    upload_payloads = [{**payload, "analysis": analysis_id}
                       for payload in valid_payloads]
    if POST_DATA:
        for payload in tqdm_notebook(upload_payloads,
                                    desc=f"Uploading results for {analysis_def.name}"):
            response = api.create_result(payload)
            upload_log.append({"analysis": analysis_def.name,
                                "uploaded_rows": len(upload_payloads),
                                "status": bool(response["exists"])})

display(pd.DataFrame(upload_log))


Created analysis with ID 34 for LifetimeDesignFrequencies, uploading 220 results...


Uploading results for LifetimeDesignFrequencies:   0%|          | 0/220 [00:00<?, ?it/s]

,analysis,uploaded_rows,status
0,LifetimeDesignFrequencies,220,True
1,LifetimeDesignFrequencies,220,True
2,LifetimeDesignFrequencies,220,True
3,LifetimeDesignFrequencies,220,True
4,LifetimeDesignFrequencies,220,True
...,...,...,...
215,LifetimeDesignFrequencies,220,True
216,LifetimeDesignFrequencies,220,True
217,LifetimeDesignFrequencies,220,True
218,LifetimeDesignFrequencies,220,True


## Retrieve And Plot Uploaded Lifetime Design Frequencies

The next cell fetches the uploaded `Lifetime Design Frequencies` analysis rows back from the backend, reconstructs the normalized result table with the results package, and renders package plots from the retrieved rows.

In [120]:
# analysis_id = 33
retrieval_api = api if "api" in globals() else ResultsAPI(api_root=BASE_URL, token=TOKEN)
results_service = ResultsService(
    repository=ApiResultsRepository(api=retrieval_api),
)
retrieved_series = results_service.get_result_series(
    lifetime_design_frequencies_analysis.analysis_name,
    filters={"analysis_id": analysis_id},
)
frequencies_analysis = LifetimeDesignFrequencies()
retrieved_frequencies = frequencies_analysis.from_results(retrieved_series)
retrieved_frequencies.head()


,x,y,series_name,turbine,metric,reference,location_id,site_id
0,INFL,0.3406,BBA01 - FA1,BBA01,FA1,INFL,435,35
1,ACTU,0.3330,BBA01 - FA1,BBA01,FA1,ACTU,435,35
2,FLEX,0.3254,BBA01 - FA1,BBA01,FA1,FLEX,435,35
3,INFL,1.0172,BBA01 - FA2,BBA01,FA2,INFL,435,35
4,ACTU,0.9619,BBA01 - FA2,BBA01,FA2,ACTU,435,35


In [121]:
location_ids = sorted({series.location_id for series in retrieved_series if series.location_id is not None})
location_lookup = results_service.get_location_frame(location_ids)

storage_diagnostics = pd.DataFrame(
    [
        {
            "short_description": series.short_description,
            "derived_metric": (
                series.short_description.rsplit(" - ", 1)[1]
                if " - " in series.short_description
                else None
            ),
            "stored_additional_data_keys": sorted(series.data_additional.keys()),
            "stored_reference_labels": series.data_additional.get("reference_labels"),
            "location_coordinates_available": {"northing", "easting"}.issubset(location_lookup.columns),
        }
        for series in retrieved_series
    ]
)

display(storage_diagnostics)
if not location_lookup.empty:
    display(location_lookup.head())


,short_description,derived_metric,stored_additional_data_keys,stored_reference_labels,location_coordinates_available
0,BBA01 - FA1,FA1,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
1,BBA01 - FA2,FA2,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
2,BBA01 - SS1,SS1,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
3,BBA01 - SS2,SS2,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
4,BBA02 - FA1,FA1,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
...,...,...,...,...,...
215,BBF04 - SS2,SS2,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
216,BBF05 - FA1,FA1,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
217,BBF05 - FA2,FA2,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True
218,BBF05 - SS1,SS1,"[analysis_kind, reference_labels, result_scope]","[INFL, ACTU, FLEX]",True


,id,title,northing,easting
194,435,BBA01,51.697000,2.805556
195,436,BBA02,51.691333,2.802167
196,437,BBA03,51.685701,2.798815
197,438,BBA04,51.679944,2.795361
198,439,BBA05,51.674258,2.791977


In [122]:
comparison_plot = results_service.plot_results(
    lifetime_design_frequencies_analysis.analysis_name,
    filters={"analysis_id": analysis_id},
    plot_type="comparison",
)
location_plot = results_service.plot_results(
    lifetime_design_frequencies_analysis.analysis_name,
    filters={"analysis_id": analysis_id},
    plot_type="location",
)
geo_plot = results_service.plot_results(
    lifetime_design_frequencies_analysis.analysis_name,
    filters={"analysis_id": analysis_id},
    plot_type="geo",
)
display(comparison_plot.notebook)
display(location_plot.notebook)
display(geo_plot.notebook)


In [123]:
plot_summary = pd.DataFrame(
    [
        {
            "retrieved_series": len(retrieved_series),
            "reference_labels_present": all(
                bool(series.data_additional.get("reference_labels"))
                for series in retrieved_series
            ),
            "comparison_plot_available": comparison_plot.notebook is not None,
            "location_plot_available": location_plot.notebook is not None,
            "geo_plot_available": geo_plot is not None and geo_plot.notebook is not None,
        }
    ]
)
display(plot_summary)


,retrieved_series,reference_labels_present,comparison_plot_available,location_plot_available,geo_plot_available
0,220,True,True,True,True
